# 🚀 DeepScalper Mission Control Plane
**Project:** FinRL-Pro_DS | **Architecture:** Mach 3 (RTX 5090)

This notebook serves as the interactive **Cockpit** for the MLOps pipeline. Use it to:
1.  **Configure:** Modify `deepscalper_unified.yaml` and experiment params.
2.  **Deploy:** Launch Training/HPO jobs to GPUHub/RunPod.
3.  **Monitor:** Watch real-time logs and process status.
4.  **Analyze:** Fetch results and update the Research Log.

## 1. Environment Setup

In [ ]:
import os
import sys
import yaml
import subprocess
import pandas as pd
from datetime import datetime
import time

# Ensure Project Root is in Path
PROJECT_ROOT = os.path.abspath("../")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
    
os.chdir(PROJECT_ROOT)
print(f"✅ Working Directory set to: {os.getcwd()}")

## 2. Configuration Management
Edit hyperparameters below or open the full YAML file.

In [ ]:
# Load current config
CONFIG_PATH = "configs/deepscalper_unified.yaml"
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

print(f"🔹 Current Architecture: {config['network']['hidden_size']} units")
print(f"🔹 Current Agents: {list(config['agents'].keys())}")
print(f"🔹 Training timesteps: {config['training']['total_timesteps']}")

In [ ]:
# 🛠️ Quick Edit: Run this cell to update Training Steps or Batch Size temporarily
config['training']['total_timesteps'] = 5000000
config['training']['batch_size'] = 4096

# Save back (Optional - Uncomment to overwrite)
# with open(CONFIG_PATH, "w") as f:
#     yaml.dump(config, f)
# print("✅ Configuration Updated!")

## 3. Mission Launch (Deployment)
Choose your mission type and target.

In [ ]:
# 🎛️ Mission Parameters
MISSION_TYPE = "hpo" # 'train' or 'hpo'
TARGET = "gpuhub"    # 'gpuhub' or 'runpod'
RUN_ID = f"DS_Mission_{datetime.now().strftime('%Y%m%d_%H%M')}"
NOTES = "Alpha HPO Sweep on Full Dataset"

HOST = "root@<LAN_HOST>" # REPLACE WITH ACTUAL HOST
KEY_PATH = "~/.ssh/id_rsa"  # REPLACE WITH ACTUAL KEY PATH

print(f"🚀 Preparing Mission: {RUN_ID}")
print(f"   Target: {TARGET} ({HOST})")

In [ ]:
# 🟢 EXECUTE LAUNCH
# This runs the bare_metal deployer script

cmd = [
    "python", "scripts/deploy_bare_metal.py",
    "--host", HOST,
    "--key", KEY_PATH,
    "--run_name", RUN_ID,
    # Add specific flags based on mission
    "--script", "scripts/tune_deepscalper.py" if MISSION_TYPE == "hpo" else "scripts/train_deepscalper_v3.py"
]

print(f"Running: {' '.join(cmd)}")
# subprocess.run(cmd) # Uncomment to Run

## 4. In-Flight Monitoring
Query the remote server for status.

In [ ]:
# Check Process Status
!python scripts/monitor_status.py --host {HOST} --key {KEY_PATH}

In [ ]:
# Stream Logs (Tail)
!python scripts/remote_cmd.py "tail -n 50 /workspace/FinRL-Pro_DS/logs/latest.log" --host {HOST} --key {KEY_PATH}

## 5. Post-Flight Analysis & Logging
Retrieve data and update the Research Log.

In [ ]:
# Fetch HPO Database or Trade Logs
!python scripts/fetch_db.py --host {HOST} --key {KEY_PATH}
print("✅ Data retrieved to local directory.")

In [ ]:
# Auto-Log to Research Journal (randd_log.md)
# Ensure you have the 'Research Logger' skill configured

log_cmd = [
    "python", ".agent/skills/research_logger/scripts/log_research.py",
    "--title", f"Mission Report: {RUN_ID}",
    "--objective", NOTES,
    "--issue", "Pending Analysis",
    "--solution", "Executed Standard Pipeline",
    "--conclusion", "Pending Verification"
]

# subprocess.run(log_cmd) # Uncomment to Log